### Load Datasets

In [ ]:
!pip install geopandas

In [ ]:
!pip install folium

In [ ]:
!pip install geodatasets


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from geodatasets import get_path
from function_06 import load_data_with_delimiters

In [ ]:
file_info = [
    ("../data/raw/FIETSTELLINGEN1723-ams.csv", ';'),    
    ("../data/raw/ongevallen_2015_2020.csv", ',')      
]

dfs = load_data_with_delimiters(file_info)

df_ams = dfs[0]
df_nl = dfs[1]

print(df_ams.head())
print(df_nl.head())


In [ ]:
 df_ams.head()

In [ ]:
 df_nl.head()

In [ ]:
#Drop unnecessary  columns
df_ams=df_ams.drop(columns=['Unnamed: 13'])
df_nl = df_nl.drop(columns=["gdb_geomattr_data"])

In [ ]:

# Rename columns
df_ams.columns = [
    "object_id", "weekday_2023", "weekday_2022", "weekday_2021", "weekday_2020",
    "weekday_2019", "weekday_2018", "weekday_2017",
    "location_code", "wkt_lng_lat", "wkt_lat_lng", "longitude", "latitude"
]



In [ ]:
#change column name
df_nl = df_nl.rename (columns={
    "id": "accident_id",
    "totaal": "total_accidents",
    "dodelijk": "fatal_accidents",
    "letsel": "injury_accidents",
    "matrieel": "damage_only_accidents",
    "shape": "geometry_wkt"
})

In [ ]:
df_ams.dtypes

In [ ]:
df_nl.dtypes

In [ ]:
import geopandas as gpd
from shapely.wkt import loads
# Parse WKT into geometries
df_nl["geometry"] = df_nl["geometry_wkt"].apply(loads)

# Create GeoDataFrame
gdf_nl = gpd.GeoDataFrame(df_nl, geometry="geometry", crs="EPSG:28992")


# Convert to projected CRS (RD New) for accurate spatial calculations- Dutch coordinate system (RD New - EPSG:28992)
gdf_nl_projected = gdf_nl.to_crs(epsg=28992)

# Calculate centroids in meters
gdf_nl_projected["centroid"] = gdf_nl_projected.geometry.centroid

#  Convert centroids back to WGS84 for mapping
gdf_nl_centroids = gdf_nl_projected.set_geometry("centroid").to_crs(epsg=4326)

# Extract latitude and longitude
gdf_nl_centroids["longitude"] = gdf_nl_centroids.geometry.x
gdf_nl_centroids["latitude"] = gdf_nl_centroids.geometry.y

print(gdf_nl_centroids.head())


In [ ]:
gdf_ams =gpd.GeoDataFrame(
    df_ams, geometry=geopandas.points_from_xy(df_ams.longitude, df_ams.latitude), crs="EPSG:4326"
)
